# 01 · Eğitim (K5: 3 seed, K2-b: Ped2)

Orijinal notebook'taki eğitimin **birebir aynısı**: aynı mimari, kayıp (0,8·MSE + 0,2·L1), Adam (lr 5e-4), CosineAnnealingWarmRestarts (T0=15, Tmult=2), veri artırma, 60 epoch, erken durdurma (sabır 15), batch 32.

Tek fark: kareler her epoch Drive'dan tek tek okunmuyor, bir kez okunup bellekte tutuluyor. İşlem aynı (gri ton, 64×64 BILINEAR), eğitim sadece çok daha hızlı.

**Bu notebook'u 3 kez çalıştırın. Her seferinde sadece `RUN` satırını değiştirin:**

| Çalıştırma | `RUN` | Çıktı | Tahmini süre (T4) |
|---|---|---|---|
| 1 | `'avenue_seed7'` | `model_seed7.pth` | ~20–40 dk |
| 2 | `'avenue_seed2024'` | `model_seed2024.pth` | ~20–40 dk |
| 3 | `'ped2'` | `model_ped2.pth` (UCSD veri setini kendisi indirir) | ~15–25 dk |
| Ablasyon 1 | `'abl_skip_yok'` | `model_abl_skip_yok.pth`: atlama bağlantısı hiç yok | ~20–40 dk |
| Ablasyon 2 | `'abl_skip_derin'` | `model_abl_skip_derin.pth`: yalnızca en derin seviyede (E3) | ~20–40 dk |
| Ablasyon 3 | `'abl_artirma_yok'` | `model_abl_artirma_yok.pth`: veri artırma kapalı | ~20–40 dk |
| Ablasyon 4 | `'abl_pencere16'` | `model_abl_pencere16.pth`: 16 karelik pencere | ~30–50 dk |

Ablasyonların hepsi seed 42 ile eğitilir. Tek farkları tablodaki değişikliktir; geri kalan her şey makaledeki modelle aynıdır.

Seed 42 modeli (`model_90plus.pth`) zaten var, onu yeniden eğitmeyin. Doğrulama bölmesi tüm seed'lerde **aynı** (seed 42 ile ayrılıyor). Seed yalnızca ağırlık başlatmayı, karıştırmayı ve veri artırmayı değiştiriyor. Böylece 02 notebook'u eşik için hep aynı 382 sekansı kullanabiliyor.

In [ ]:
RUN = 'avenue_seed7'      # 'avenue_seed7' | 'avenue_seed2024' | 'ped2' | 'abl_skip_yok' | 'abl_skip_derin' | 'abl_artirma_yok' | 'abl_pencere16'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, glob, random, warnings, time, shutil, zipfile, tarfile, urllib.request
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn

import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CFG:
    PROJ       = '/content/drive/MyDrive/AnomalyProject'
    TRAIN_DIR  = f'{PROJ}/frames/avenue_train'
    CACHE_DIR  = f'{PROJ}/revizyon_sonuclari/cache'      # 1. turda oluşan önbellek (avenue_train.npy)
    PED2_CACHE = f'{PROJ}/data/ped2_cache'
    PED2_URL   = 'http://www.svcl.ucsd.edu/projects/anomaly/UCSD_Anomaly_Dataset.tar.gz'
    LOG_DIR    = f'{PROJ}/egitim_kayitlari'
    TMP        = '/content/tmp'

    IMG_SIZE=64; SEQ_LEN=8; SEQ_STEP=2
    EPOCHS=60; BATCH_SIZE=32; LR=5e-4; WEIGHT_DECAY=1e-5
    ES_PATIENCE=15; T_0=15; T_MULT=2; ETA_MIN=1e-7
    NUM_WORKERS=2
    VAL_SPLIT_SEED = 42          # doğrulama bölmesi tüm seed'lerde aynı

SETTINGS = {
    'avenue_seed7'   : dict(data='avenue', seed=7,    out='model_seed7.pth'),
    'avenue_seed2024': dict(data='avenue', seed=2024, out='model_seed2024.pth'),
    'avenue_seed42'  : dict(data='avenue', seed=42,   out='model_seed42_yeniden.pth'),  # gerekmez, kontrol için
    'ped2'           : dict(data='ped2',   seed=42,   out='model_ped2.pth'),
    # Ablasyonlar (Ö1, Ö5): yalnızca belirtilen ayar değişir
    'abl_skip_yok'   : dict(data='avenue', seed=42, out='model_abl_skip_yok.pth',    skip=(False, False, False)),
    'abl_skip_derin' : dict(data='avenue', seed=42, out='model_abl_skip_derin.pth',  skip=(False, False, True)),
    'abl_artirma_yok': dict(data='avenue', seed=42, out='model_abl_artirma_yok.pth', augment=False),
    'abl_pencere16'  : dict(data='avenue', seed=42, out='model_abl_pencere16.pth',   seq_len=16),
}
S = SETTINGS[RUN]
SEED = S['seed']; MODEL_PATH = f"{CFG.PROJ}/{S['out']}"
SKIP = tuple(S.get('skip', (True, True, True))); AUGMENT = S.get('augment', True)
CFG.SEQ_LEN = S.get('seq_len', 8)
os.makedirs(CFG.LOG_DIR, exist_ok=True); os.makedirs(CFG.TMP, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Çalıştırma: {RUN} | seed={SEED} | çıktı={MODEL_PATH} | cihaz={DEVICE}')

In [ ]:
EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', str(s))]

def list_frames(folder, natural=True):
    fs = [f for f in os.listdir(folder) if f.lower().endswith(EXTS)]
    fs = sorted(fs, key=natural_key) if natural else sorted(fs)
    return [os.path.join(folder, f) for f in fs]

def to_gray64(img):
    # Ana notebook'taki load_frame ile aynı: gri ton + 64x64 BILINEAR, [0,1]
    if not isinstance(img, Image.Image): img = Image.fromarray(img)
    return np.asarray(img.convert('L').resize((CFG.IMG_SIZE, CFG.IMG_SIZE), Image.BILINEAR),
                      np.float32) / 255.0

def load_stack(paths, cache_name=None, cache_dir=None):
    cache_dir = cache_dir or CFG.CACHE_DIR
    if cache_name:
        cp = os.path.join(cache_dir, cache_name + '.npy')
        if os.path.exists(cp):
            arr = np.load(cp)
            if len(arr) == len(paths): return arr
    arr = np.stack([to_gray64(Image.open(p)) for p in paths])
    if cache_name:
        os.makedirs(cache_dir, exist_ok=True); np.save(os.path.join(cache_dir, cache_name + '.npy'), arr)
    return arr

def cached_array(name, fn, cache_dir=None):
    cp = os.path.join(cache_dir or CFG.CACHE_DIR, name + '.npy')
    if os.path.exists(cp): return np.load(cp)
    arr = fn(); os.makedirs(os.path.dirname(cp), exist_ok=True); np.save(cp, arr); return arr

def frames_from_avi(path):
    import cv2
    cap, out = cv2.VideoCapture(path), []
    while True:
        ok, f = cap.read()
        if not ok: break
        out.append(to_gray64(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    cap.release()
    return np.stack(out)

def fetch(url, dst, tries=3):
    # Dosyayı indirir; zaten varsa atlar. Başarısızsa False döner.
    if os.path.exists(dst) and os.path.getsize(dst) > 0: return True
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for k in range(tries):
        try:
            urllib.request.urlretrieve(url, dst + '.part'); os.replace(dst + '.part', dst); return True
        except Exception as e:
            print(f'   indirme hatası ({k+1}/{tries}) {os.path.basename(url)}: {e}')
            time.sleep(3)
    return False

# ── UCSD Ped2 ─────────────────────────────────────────────────────────
def parse_ped_gt(m_path):
    txt = open(m_path, encoding='latin-1').read()
    out = []
    for body in re.findall(r'gt_frame\s*=\s*\[([^\]]*)\]', txt):
        fr = []
        for a, b in re.findall(r'(\d+)\s*:\s*(\d+)', body): fr += list(range(int(a), int(b) + 1))
        rest = re.sub(r'\d+\s*:\s*\d+', ' ', body)
        fr += [int(x) for x in re.findall(r'\d+', rest)]
        out.append(sorted(set(fr)))
    return out

def prepare_ped2():
    cache = CFG.PED2_CACHE
    if os.path.exists(f'{cache}/done.json'):
        meta = json.load(open(f'{cache}/done.json'))
        tr = [np.load(f'{cache}/train_{i:02d}.npy') for i in range(meta['n_train'])]
        te = [np.load(f'{cache}/test_{i:02d}.npy') for i in range(meta['n_test'])]
        lb = [np.load(f'{cache}/label_{i:02d}.npy') for i in range(meta['n_test'])]
        return tr, te, lb
    tgz = f'{CFG.TMP}/UCSD_Anomaly_Dataset.tar.gz'
    print('UCSD veri seti indiriliyor (~700 MB)...')
    if not fetch(CFG.PED2_URL, tgz): raise RuntimeError('Ped2 indirilemedi: ' + CFG.PED2_URL)
    ex = f'{CFG.TMP}/ucsd'
    if not os.path.exists(ex):
        with tarfile.open(tgz) as t: t.extractall(ex)
    root = next(r for r, ds, _ in os.walk(ex) if os.path.basename(r) == 'UCSDped2')
    trd = sorted([d for d in glob.glob(f'{root}/Train/*') if re.fullmatch(r'Train\d+', os.path.basename(d))], key=natural_key)
    ted = sorted([d for d in glob.glob(f'{root}/Test/*') if re.fullmatch(r'Test\d+', os.path.basename(d))], key=natural_key)
    gt  = parse_ped_gt(glob.glob(f'{root}/Test/*.m')[0])
    os.makedirs(cache, exist_ok=True)
    tr, te, lb = [], [], []
    for i, d in enumerate(trd):
        a = load_stack(list_frames(d)); np.save(f'{cache}/train_{i:02d}.npy', a); tr.append(a)
    for i, d in enumerate(ted):
        a = load_stack(list_frames(d)); l = np.zeros(len(a), int)
        for f in gt[i]:
            if 1 <= f <= len(a): l[f - 1] = 1
        np.save(f'{cache}/test_{i:02d}.npy', a); np.save(f'{cache}/label_{i:02d}.npy', l); te.append(a); lb.append(l)
    json.dump({'n_train': len(tr), 'n_test': len(te)}, open(f'{cache}/done.json', 'w'))
    print(f'Ped2 hazır: {len(tr)} eğitim, {len(te)} test videosu')
    return tr, te, lb

In [ ]:
class CB3(nn.Module):
    def __init__(self, ic, oc, k=(3,3,3), s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(nn.Conv3d(ic, oc, k, s, p, bias=False),
                                 nn.BatchNorm3d(oc), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.net(x)

class AE3D(nn.Module):
    # skip = (E1, E2, E3) seviyelerinde atlama bağlantısı var mı. Varsayılan: hepsi (makaledeki model).
    def __init__(self, skip=(True, True, True)):
        super().__init__()
        self.skip = tuple(skip); s1, s2, s3 = self.skip
        self.e1=nn.Sequential(CB3(1,32),CB3(32,32));     self.p1=nn.MaxPool3d((1,2,2),(1,2,2))
        self.e2=nn.Sequential(CB3(32,64),CB3(64,64));    self.p2=nn.MaxPool3d(2,2)
        self.e3=nn.Sequential(CB3(64,128),CB3(128,128)); self.p3=nn.MaxPool3d(2,2)
        self.bn=nn.Sequential(CB3(128,128),CB3(128,64),CB3(64,128),nn.Dropout3d(0.4))
        self.u3=nn.Upsample(scale_factor=(2,2,2),mode='trilinear',align_corners=False)
        self.d3=nn.Sequential(CB3(256 if s3 else 128,128),CB3(128,64))
        self.u2=nn.Upsample(scale_factor=(2,2,2),mode='trilinear',align_corners=False)
        self.d2=nn.Sequential(CB3(128 if s2 else 64,64),CB3(64,32))
        self.u1=nn.Upsample(scale_factor=(1,2,2),mode='trilinear',align_corners=False)
        self.d1=nn.Sequential(CB3(64 if s1 else 32,32),CB3(32,32))
        self.out=nn.Sequential(nn.Conv3d(32,1,1),nn.Sigmoid())
        for m in self.modules():
            if isinstance(m, nn.Conv3d): nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
            elif isinstance(m, nn.BatchNorm3d): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def forward(self,x):
        e1=self.e1(x); e1p=self.p1(e1)
        e2=self.e2(e1p); e2p=self.p2(e2)
        e3=self.e3(e2p); e3p=self.p3(e3)
        b=self.bn(e3p)
        s1, s2, s3 = self.skip
        d=self.u3(b);  d=self.d3(torch.cat([d,e3],1) if s3 else d)
        d=self.u2(d);  d=self.d2(torch.cat([d,e2],1) if s2 else d)
        d=self.u1(d);  d=self.d1(torch.cat([d,e1],1) if s1 else d)
        return self.out(d)

def win_idx(n, step, stride=1):
    # 8 karelik pencereler: başlangıç adımı `step`, pencere içi kare aralığı `stride`
    span = (CFG.SEQ_LEN - 1) * stride + 1
    return [[s + t * stride for t in range(CFG.SEQ_LEN)] for s in range(0, n - span + 1, step)]

## Veri

In [ ]:
if S['data'] == 'avenue':
    # Orijinal notebook ile aynı: düz klasör, sorted() sırası, ardışık 8 kare, adım 2
    paths  = list_frames(CFG.TRAIN_DIR, natural=False)
    videos = [load_stack(paths, 'avenue_train')]
else:
    videos, _, _ = prepare_ped2()

class SeqDataset(Dataset):
    def __init__(self, videos, augment=True):
        self.videos = [torch.from_numpy(v) for v in videos]
        self.items  = [(vi, w) for vi, v in enumerate(videos) for w in win_idx(len(v), CFG.SEQ_STEP, 1)]
        self.augment = augment
        self.aug = T.Compose([T.RandomHorizontalFlip(0.5), T.RandomVerticalFlip(0.2),
                              T.RandomAffine(degrees=5, translate=(0.05,0.05), scale=(0.95,1.05)),
                              T.GaussianBlur(3, sigma=(0.1, 1.5))])
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        vi, w = self.items[i]
        frames = [self.videos[vi][t].unsqueeze(0) for t in w]       # [1,H,W]
        if self.augment:                                           # aynı dönüşüm tüm karelere
            state = torch.get_rng_state(); out = []
            for f in frames:
                torch.set_rng_state(state); out.append(self.aug(f))
            frames = out
        return torch.stack(frames, dim=1)                          # [1,T,H,W]

ds = SeqDataset(videos, augment=AUGMENT)
n = len(ds); val_n = max(1, int(n * 0.10))
tr_sub, val_sub = torch.utils.data.random_split(ds, [n - val_n, val_n],
                                                generator=torch.Generator().manual_seed(CFG.VAL_SPLIT_SEED))
DL = dict(num_workers=CFG.NUM_WORKERS, pin_memory=True,
          persistent_workers=CFG.NUM_WORKERS > 0, prefetch_factor=2 if CFG.NUM_WORKERS > 0 else None)
train_loader = DataLoader(tr_sub, batch_size=CFG.BATCH_SIZE, shuffle=True, drop_last=True, **DL)
val_loader   = DataLoader(val_sub, batch_size=CFG.BATCH_SIZE, shuffle=False, **DL)
print(f'{sum(map(len, videos))} kare → {n} sekans | eğitim {len(tr_sub)} | doğrulama {len(val_sub)}')

## Eğitim

In [ ]:
model = AE3D(skip=SKIP).to(DEVICE)
print(f'Model: atlama bağlantıları (E1,E2,E3) = {SKIP} | pencere = {CFG.SEQ_LEN} | veri artırma = {AUGMENT} | parametre = {sum(p.numel() for p in model.parameters()):,}')
class Loss(nn.Module):
    def __init__(self): super().__init__(); self.mse = nn.MSELoss(); self.l1 = nn.L1Loss()
    def forward(self, p, t): return 0.8 * self.mse(p, t) + 0.2 * self.l1(p, t)
criterion = Loss().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=CFG.T_0, T_mult=CFG.T_MULT, eta_min=CFG.ETA_MIN)

best_val, es, log = float('inf'), 0, {'train': [], 'val': [], 'lr': []}
t_start = time.time()
for epoch in range(1, CFG.EPOCHS + 1):
    t0 = time.time(); model.train(); tl = 0.0
    for bi, x in enumerate(train_loader):
        x = x.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), x); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(epoch - 1 + bi / len(train_loader))
        tl += loss.item()
    tl /= len(train_loader)
    model.eval(); vl = 0.0
    with torch.no_grad():
        for x in val_loader: x = x.to(DEVICE); vl += criterion(model(x), x).item()
    vl /= len(val_loader)
    log['train'].append(tl); log['val'].append(vl); log['lr'].append(optimizer.param_groups[0]['lr'])
    if vl < best_val - 1e-6:
        best_val, es = vl, 0; torch.save(model.state_dict(), MODEL_PATH); mark = ' ★'
    else:
        es += 1; mark = ''
    print(f'{epoch:>3} | eğitim {tl:.6f} | doğrulama {vl:.6f} | lr {log["lr"][-1]:.2e} | ES {es:>2} | {time.time()-t0:.0f}s{mark}')
    if es >= CFG.ES_PATIENCE:
        print(f'Erken durdurma: epoch {epoch}'); break

log.update(run=RUN, seed=SEED, best_val=best_val, best_epoch=int(np.argmin(log['val'])) + 1,
           epochs_run=len(log['val']), minutes=(time.time() - t_start) / 60, model=MODEL_PATH)
log.update(skip=list(SKIP), seq_len=CFG.SEQ_LEN, augment=AUGMENT)
json.dump(log, open(f'{CFG.LOG_DIR}/{RUN}.json', 'w'), indent=1)
json.dump({'skip': list(SKIP), 'seq_len': CFG.SEQ_LEN, 'augment': AUGMENT, 'seed': SEED, 'run': RUN},
          open(MODEL_PATH.replace('.pth', '.json'), 'w'))   # 03 notebook'u modeli doğru kurmak için okur
print(f"\n✅ Bitti: en iyi doğrulama kaybı {best_val:.6f} (epoch {log['best_epoch']}) | {log['minutes']:.1f} dk")
print('Kaydedildi:', MODEL_PATH)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(log['train'])+1), log['train'], label='Eğitim', lw=2)
ax.plot(range(1, len(log['val'])+1), log['val'], label='Doğrulama', lw=2, ls='--')
ax.axvline(log['best_epoch'], color='gray', ls=':', label=f"En iyi epoch: {log['best_epoch']}")
ax.set_xlabel('Epoch'); ax.set_ylabel('Kayıp (0,8·MSE + 0,2·L1)'); ax.set_title(f'Eğitim eğrisi — {RUN}')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{CFG.LOG_DIR}/{RUN}.png', dpi=200); plt.show()